# NFPP Sodium-Ion ESS Research Pipeline

This notebook implements the complete research pipeline for the Sodium Iron Pyrophosphate (NFPP) battery system.

In [ ]:
import os
import subprocess
import sys
import pandas as pd
from IPython.display import HTML, display, IFrame

os.environ['PYTHONPATH'] = os.environ.get('PYTHONPATH', '') + ':' + os.getcwd()

import pybamm
import numpy as np
import matplotlib.pyplot as plt
print("Environment initialized.")

## Stage 1: Cell Verification
Verify the base parameter set against literature-referenced material properties.

In [ ]:
from nfpp_sodium_ion.src.calibration.verify_parameters import verify
from nfpp_sodium_ion.src.cell_parameters.cell_alpha import get_parameter_values

print("Stage 1: Running Cell Verification...")
verify()
base_params = get_parameter_values()

## Stage 2: Cell Optimization
Two-step optimization: Electrolyte (Cost) and Sensitivity-Driven Hessian Optimization.

In [ ]:
from src.optimization_pipeline.optimize import NFPPoptimizer
print("Stage 2: Running Optimization...")
optimizer = NFPPoptimizer()
optimization_results = optimizer.run_optimization()

## Stage 3: Stability Validation
Comprehensive performance evaluation and resistance profile generation.

In [ ]:
from src.optimization_pipeline.validate import StabilityValidator
print("Stage 3: Running Stability Validation...")
validator = StabilityValidator(base_params)
results = validator.validate_electrochemical_performance(optimized_subset=optimization_results)

print("\nPerformance Metrics Summary:")
metrics_df = pd.DataFrame([{
    'Metric': k, 
    'Value': f"{v:.4f}" if isinstance(v, float) else v
} for k, v in results.items() if k not in ['merged_params', 'resistance_profile', 'envelope_sweep']])
display(metrics_df)

# Export for MATLAB
validator.export_to_matlab(results, output_path="src/control_systems/optimized_params.mat")

## Stage 4: BMS Controller Validation
Analyze stability and response characteristics using MATLAB's professional reporting tools.

In [ ]:
print("Stage 4: Generating MATLAB BMS Validation Report...")
matlab_cmd = "matlab -nodisplay -nosplash -nodesktop -r \"publish('src/control_systems/validate_controller.m'); quit;\""

try:
    subprocess.check_call(["which", "matlab"], stdout=subprocess.DEVNULL)
    subprocess.run(matlab_cmd, shell=True, check=True)
    print("MATLAB Report Generated.")
    
    report_path = "src/control_systems/html/validate_controller.html"
    if os.path.exists(report_path):
        display(IFrame(src=report_path, width='100%', height=800))
except (subprocess.CalledProcessError, FileNotFoundError):
    print("MATLAB not installed or execution failed.")